# Machine Learning Task 15 — Production Recommender Architecture
### Retrieval → Scoring → Re-Ranking, ANN serving, latency budgets, monitoring & A/B loop

**Dataset:** MovieLens `ml-latest-small` (auto-downloaded in the first code cell).

**How this notebook is organised:** one section per phase of `task15-readme.md` (Phase 1–8), then the 5 exercises, then the deliverables checklist. Run it top to bottom.

> **Heads-up on Tasks 11–14:** the models from those tasks live in your other notebooks, so this notebook **re-creates small stand-ins** (SVD embeddings for Task 12, a NeuMF scorer for Task 14, genre features for Task 11). If you want to plug in your own trained Task 14 models, just replace the `neumf` model and the SVD embeddings — every other cell only depends on the interfaces (`neumf_score(u, items)` and an item-embedding matrix).

## 0. Setup & Data
Install once (uncomment). `faiss-cpu` and `tensorflow` are the only heavy dependencies.

In [ ]:
# !pip install faiss-cpu tensorflow scipy pandas matplotlib
import os, io, time, zipfile, urllib.request, warnings, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

import tensorflow as tf
import faiss

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED); random.seed(SEED)
print("TensorFlow", tf.__version__, "| FAISS", faiss.__version__)

RESULTS = {}   # headline numbers are collected here for the final production-readiness summary

In [ ]:
DATA_DIR = "ml-latest-small"
if not os.path.exists(os.path.join(DATA_DIR, "ratings.csv")):
    print("Downloading MovieLens ml-latest-small ...")
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(".")

ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
movies  = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"))

# ---- contiguous ids ----
user_raw = np.sort(ratings.userId.unique()); user2idx = {u: i for i, u in enumerate(user_raw)}
item_raw = np.sort(movies.movieId.unique()); item2idx = {m: i for i, m in enumerate(item_raw)}
ratings["u"] = ratings.userId.map(user2idx); ratings["i"] = ratings.movieId.map(item2idx)
movies["i"] = movies.movieId.map(item2idx)
movies = movies.sort_values("i").reset_index(drop=True)
assert (movies.i.values == np.arange(len(movies))).all()
n_users, n_items = len(user_raw), len(item_raw)
titles = movies.title.values
genres_str = movies.genres.values
print(f"{n_users} users | {n_items} items | {len(ratings)} ratings")

# ---- item side features: release year + genres (Task 11 style) ----
year = movies.title.str.extract(r"\((\d{4})\)\s*$")[0].astype(float)
year = year.fillna(year.median()).values
item_year = year
all_genres = sorted({g for s in movies.genres for g in s.split("|")} - {"(no genres listed)"})
g2idx = {g: k for k, g in enumerate(all_genres)}; G = len(all_genres)
item_genres = np.zeros((n_items, G), np.float32)
primary_genre = -np.ones(n_items, int)
for r, gs in enumerate(genres_str):
    gl = [g for g in gs.split("|") if g in g2idx]
    for g in gl: item_genres[r, g2idx[g]] = 1
    if gl: primary_genre[r] = g2idx[gl[0]]
year_norm = (item_year - item_year.min()) / (item_year.max() - item_year.min())
item_feats = np.hstack([item_genres, year_norm[:, None]]).astype(np.float32)
freshness = np.exp(-(item_year.max() - item_year) / 10.0)          # 1.0 = newest, decays over ~10 years
print("genres:", G, "| year range:", int(item_year.min()), "-", int(item_year.max()))

In [ ]:
# ---- implicit-feedback split: positives = rating >= 4, per-user chronological 80/20 ----
pos = ratings[ratings.rating >= 4.0].sort_values(["u", "timestamp"]).copy()
pos["pct"] = pos.groupby("u").cumcount() / pos.groupby("u")["u"].transform("count")
train_pos = pos[pos.pct < 0.8].copy()
test_pos  = pos[pos.pct >= 0.8].copy()

# "seen" = everything the user interacted with before their test period (never re-recommend these)
cut = test_pos.groupby("u").timestamp.min()
ratings["cut"] = ratings.u.map(cut).fillna(np.inf)
train_ratings = ratings[ratings.timestamp < ratings.cut]
seen_mask = np.zeros((n_users, n_items), bool)
seen_mask[train_ratings.u.values, train_ratings.i.values] = True
seen_mask[train_pos.u.values, train_pos.i.values] = True
n_seen = seen_mask.sum(1)

test_items = {u: {i for i in s if not seen_mask[u, i]} for u, s in test_pos.groupby("u").i.apply(set).items()}
eval_users = np.array([u for u, s in test_items.items() if len(s) >= 3])
rng = np.random.default_rng(SEED)
sample_users = rng.choice(eval_users, min(150, len(eval_users)), replace=False)

# ---- popularity (train positives) and head/tail split ----
pop = np.bincount(train_pos.i.values, minlength=n_items)
order_pop = np.argsort(-pop)
is_tail = np.ones(n_items, bool); is_tail[order_pop[: int(0.2 * n_items)]] = False   # top-20% items = head
log_pop = np.log1p(pop)

# ---- user side features (computed from the TRAIN period only) ----
user_genre = np.zeros((n_users, G), np.float32)
np.add.at(user_genre, train_pos.u.values, item_genres[train_pos.i.values])
user_genre /= np.maximum(user_genre.sum(1, keepdims=True), 1)
cnt = np.bincount(train_pos.u.values, minlength=n_users)
user_cnt = np.log1p(cnt) / np.log1p(cnt.max())
user_mean = train_ratings.groupby("u").rating.mean().reindex(range(n_users)).fillna(3.5).values / 5.0
user_feats = np.hstack([user_genre, user_cnt[:, None], user_mean[:, None]]).astype(np.float32)

print(f"train positives: {len(train_pos)} | test positives: {len(test_pos)} | eval users: {len(eval_users)}")
print(f"tail items (outside top-20% by popularity): {is_tail.mean():.0%} of catalog")

In [ ]:
# ---------------- shared helpers: top-k and ranking metrics ----------------
def topk_from_scores(S, k):
    idx = np.argpartition(-S, k, axis=1)[:, :k]
    part = np.take_along_axis(S, idx, axis=1)
    return np.take_along_axis(idx, np.argsort(-part, axis=1), axis=1)

def ndcg_at_k(rec, rel, k=10):
    rec = list(rec)[:k]
    dcg = sum(1.0 / np.log2(r + 2) for r, i in enumerate(rec) if i in rel)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(rel), k)))
    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(rec, rel, k=100):
    return len(set(list(rec)[:k]) & rel) / len(rel)

def hit_at_k(rec, rel, k=10):
    return float(len(set(list(rec)[:k]) & rel) > 0)

def evaluate_recs(rec_dict, users, k_ndcg=10, k_recall=100):
    nd = [ndcg_at_k(rec_dict[u], test_items[u], k_ndcg) for u in users]
    hr = [hit_at_k(rec_dict[u], test_items[u], k_ndcg) for u in users]
    rc = [recall_at_k(rec_dict[u], test_items[u], k_recall) for u in users]
    return {f"NDCG@{k_ndcg}": np.mean(nd), f"Hit@{k_ndcg}": np.mean(hr), f"Recall@{k_recall}": np.mean(rc)}

# popularity baseline (sanity floor for every model below)
pop_rank = {u: order_pop[~seen_mask[u][order_pop]][:100] for u in eval_users}
base_pop = evaluate_recs(pop_rank, eval_users)
print("Popularity baseline:", {k: round(v, 4) for k, v in base_pop.items()})

---
# Phase 1 — The Three-Stage Architecture

### 1.1 The pipeline

In [ ]:
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(14, 3.8)); ax.set_xlim(0, 14); ax.set_ylim(0, 4); ax.axis("off")
stages = [
    ("Full catalog\n~9.7K movies here\n(millions in prod)", 0.2,  "#e6e6e6", 2.3),
    ("1. CANDIDATE GENERATION\ntwo-tower + ANN (FAISS)\nfast, coarse\nmillions -> hundreds", 3.1, "#cfe8ff", 3.0),
    ("2. SCORING\nNeuMF / Wide&Deep\nslow, precise\nhundreds -> ranked", 6.7, "#d6f5d6", 2.8),
    ("3. RE-RANKING\nMMR diversity, freshness,\nbusiness rules, fairness\n-> final top-N", 10.0, "#ffe9c2", 3.4),
]
for text, x, c, w in stages:
    ax.add_patch(FancyBboxPatch((x, 0.9), w, 2.2, boxstyle="round,pad=0.05", fc=c, ec="black"))
    ax.text(x + w / 2, 2.0, text, ha="center", va="center", fontsize=10)
for x0, x1 in [(2.5, 3.1), (6.1, 6.7), (9.5, 10.0)]:
    ax.annotate("", xy=(x1, 2.0), xytext=(x0, 2.0), arrowprops=dict(arrowstyle="->", lw=2))
ax.text(4.6, 0.45, "~10 ms", ha="center"); ax.text(8.1, 0.45, "~60 ms", ha="center"); ax.text(11.7, 0.45, "~10 ms", ha="center")
ax.set_title("Three-stage production recommender (example 100 ms budget)")
plt.tight_layout(); plt.show()

### 1.2 Why each stage exists

**Candidate generation must be fast and may be coarse.** It runs over the *entire* catalog on every request, so the per-item cost must be tiny — a dot product against a precomputed embedding, served from an ANN index. It's OK if the shortlist isn't perfectly ordered; its only job is **high recall** (get the truly good items into the shortlist).

**Scoring can be slow and precise.** It only sees the few hundred items retrieval handed over, so we can afford a heavy model with **cross-features** (user and item interact inside the network — e.g. NeuMF's MLP branch from Task 14). Running that same model over millions of items per request would blow any latency budget.

**Re-ranking is a separate final stage** because list-level goals ("don't show two near-identical movies back to back", "boost this week's promoted creators", "make sure new/small creators get exposure", "never show something already watched") are:
- **list-level**, not item-level — an item's value depends on what else is on the list, which a pointwise training loss can't express;
- **brittle as labels** — encoding them into training data means retraining every time product changes a rule;
- **trivial as a final pass** — a few lines of code that product can change without touching the model.

### 1.3 Mapping Tasks 11–14 onto the architecture

| Earlier task / model | Role in the 3-stage pipeline | Why |
|---|---|---|
| **Task 11** content-based (genre/TF-IDF cosine) | **Retrieval** (extra candidate source) | Cheap, works for brand-new items (cold-start), no interaction history needed |
| **Task 12** matrix factorization / item embeddings / item-item CF | **Retrieval** (embeddings -> ANN) | Fast dot products over precomputable embeddings |
| **Task 14** softmax DNN (user tower -> softmax over items) | **Retrieval** — its item-side weights *are* the candidate embeddings | Structurally a two-tower model; this is what Phase 3 builds |
| **Task 14** NCF / **NeuMF**, **Wide & Deep** | **Scoring** | User and item interact inside the network -> can't be pre-indexed, but is far more accurate on a shortlist |
| **Task 14** sequence-aware model | **Retrieval or Scoring** | As a query tower over recent-watch sequence (retrieval) or as extra features (scoring) |
| **Task 13** diversity / novelty / coverage metrics | **Re-ranking** objective + monitoring | Same ILD metric is used inside MMR and in dashboards |
| **Task 13** NDCG/MAP, A/B framework | **Monitoring & rollout loop** | Offline gate -> online experiment (Phase 8) |

---
# Phase 2 — Candidate Generation & Similarity Measures Revisited

Three formulas:

- `cosine(q,x) = <q,x> / (||q|| ||x||)`
- `dot(q,x) = <q,x>`
- `euclidean(q,x) = ||q - x||`

**Key identity:** if embeddings are L2-normalised, dot = cosine, and `1/2 ||q-x||^2 = 1 - <q,x>` — so all three give the *same ranking*. The measures only disagree when norms differ, and **norm is where popularity leaks in**: popular items get more gradient signal during training, so they tend to end up with larger-norm embeddings. Dot product rewards norm; cosine ignores it; Euclidean punishes a norm mismatch.

First, a Task-12-style embedding stand-in (SVD of the implicit user-item matrix, d=32), so we can compare the measures *before* we train the two-tower model. We repeat the comparison on the two-tower embeddings right after Phase 3.

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from scipy.stats import spearmanr

D = 32
M = csr_matrix((np.ones(len(train_pos), np.float32), (train_pos.u.values, train_pos.i.values)), shape=(n_users, n_items))
U_svd, s_svd, Vt_svd = svds(M, k=D)
svd_item_emb = (Vt_svd.T * np.sqrt(s_svd)).astype(np.float32)           # item embeddings (n_items x D)

norms = np.linalg.norm(svd_item_emb, axis=1)
print("Spearman(norm, popularity) over items with >=1 interaction:",
      round(spearmanr(norms[pop > 0], pop[pop > 0])[0], 3))

# ---- verify the normalisation identity numerically ----
q = svd_item_emb[0]; X = svd_item_emb[1:200]
qn = q / np.linalg.norm(q); Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
lhs = 0.5 * np.linalg.norm(qn - Xn, axis=1) ** 2
rhs = 1 - Xn @ qn
print("identity 1/2||q-x||^2 == 1-<q,x> on normalised vectors:", np.allclose(lhs, rhs, atol=1e-5))

In [ ]:
def compare_measures(emb, q_idx, k=10):
    q = emb[q_idx]
    dot = emb @ q
    cos = dot / (np.linalg.norm(emb, axis=1) * np.linalg.norm(q) + 1e-12)
    euc = -np.linalg.norm(emb - q, axis=1)                    # negate so "bigger = closer"
    out = {}
    for name, sc in [("dot", dot), ("cosine", cos), ("euclid", euc)]:
        sc = sc.copy(); sc[q_idx] = -np.inf
        out[name] = np.argsort(-sc)[:k]
    return out

def comparison_table(emb, q_idx, k=10):
    tops = compare_measures(emb, q_idx, k)
    ids = sorted(set().union(*[set(v) for v in tops.values()]))
    rows = []
    for i in ids:
        rows.append({"title": titles[i], "train_pop": int(pop[i]), "emb_norm": round(float(np.linalg.norm(emb[i])), 2),
                     **{f"rank_{m}": (int(np.where(tops[m] == i)[0][0]) + 1 if i in tops[m] else "-") for m in tops}})
    df = pd.DataFrame(rows)
    df["_best"] = df[["rank_dot", "rank_cosine", "rank_euclid"]].replace("-", 99).astype(int).min(axis=1)
    return df.sort_values("_best").drop(columns="_best").reset_index(drop=True), tops

q_idx = int(np.where(titles == "Toy Story (1995)")[0][0])
tbl, tops = comparison_table(svd_item_emb, q_idx)
print("Query:", titles[q_idx])
display(tbl)

only_one = {m: [i for i in tops[m] if all(i not in tops[o] for o in tops if o != m)] for m in tops}
for m, lst in only_one.items():
    print(f"Only under {m:7s}: {[(titles[i], int(pop[i])) for i in lst]}")

In [ ]:
# Aggregate view over many queries: how popular are the retrieved items under each measure?
cand_q = np.where(pop >= 30)[0]
qs = rng.choice(cand_q, min(200, len(cand_q)), replace=False)
mean_logpop = {"dot": [], "cosine": [], "euclid": []}
for qi in qs:
    t = compare_measures(svd_item_emb, qi)
    for m in t: mean_logpop[m].append(log_pop[t[m]].mean())
agg = {m: float(np.mean(v)) for m, v in mean_logpop.items()}
print(f"Mean log(1+popularity) of top-10, averaged over {len(qs)} queries:", {m: round(v, 2) for m, v in agg.items()})
plt.bar(agg.keys(), agg.values(), color=["#d62728", "#1f77b4", "#2ca02c"])
plt.ylabel("mean log(1+pop) of retrieved items"); plt.title("Popularity bias of each similarity measure (SVD embeddings)")
plt.show()

**Reading the table / chart (Phase 2 written explanation):**
- **Dot product** pulls in items with large embedding norm — i.e. the heavily-rated blockbusters — even when their *direction* is only loosely related to the query. This is the Task 12 popularity-bias finding reappearing at the retrieval stage. If the chart shows dot's bar is the tallest, that's exactly this effect.
- **Cosine** ignores norm, so it surfaces items that point the same way semantically, including rarer, more niche titles (lower `train_pop`), but it can also lift noisy low-signal items whose small embeddings happen to align.
- **Euclidean** needs both direction *and* magnitude to match, so it prefers items with a norm similar to the query's — a middle ground that tends to favour items of similar popularity to the query itself.
- Items that appear under only one measure (printed above) are the ones where norm and direction disagree — compare their `train_pop` and `emb_norm` columns to the others.

---
# Phase 3 — Two-Tower Retrieval Models

**Architecture.** A *query tower* maps user ID (+ side features: genre profile, activity, mean rating) to a `d`-dim embedding; a *candidate tower* maps item ID (+ genres, release year) to a `d`-dim embedding. The score is just `dot(query_emb, item_emb)`.

**Why this is retrieval-friendly.** The towers only meet at the final dot product, so the **item embeddings can be computed once, offline, and indexed** — before any query exists. At request time only the (cheap) query tower runs, then the index does the rest.

**Contrast with NeuMF (Task 14).** NeuMF concatenates user and item representations *inside* its MLP branch, so every prediction needs the (user, item) pair as joint input. You can't precompute anything per item that's independent of the user, so you can't index it — it's a *scoring* model, not a retrieval model. (Phase 5 uses it exactly that way.)

**Training.** In-batch sampled softmax: for a batch of (user, positive item) pairs, every *other* item in the batch acts as a negative for each user. Because popular items appear in batches more often, we subtract `log(sampling probability)` from the logits (logQ correction) so the model isn't punished for the sampling bias. Accidental duplicates of the same item in a batch are masked out. (Note: I use a linear output layer rather than the ReLU shown in the readme's sketch — a ReLU on the final embedding zeroes out half the dimensions and hurts a dot-product model.)

In [ ]:
class QueryTower(tf.keras.Model):
    def __init__(self, n_users, d, user_feats):
        super().__init__()
        self.user_feats = tf.constant(user_feats, tf.float32)
        self.embedding = tf.keras.layers.Embedding(n_users, d)
        self.hidden = tf.keras.layers.Dense(64, activation="relu")
        self.out = tf.keras.layers.Dense(d)
    def call(self, user_ids):
        x = tf.concat([self.embedding(user_ids), tf.gather(self.user_feats, user_ids)], axis=-1)
        return self.out(self.hidden(x))

class CandidateTower(tf.keras.Model):
    def __init__(self, n_items, d, item_feats):
        super().__init__()
        self.item_feats = tf.constant(item_feats, tf.float32)
        self.embedding = tf.keras.layers.Embedding(n_items, d)
        self.hidden = tf.keras.layers.Dense(64, activation="relu")
        self.out = tf.keras.layers.Dense(d)
    def _forward(self, id_vec, feats):
        return self.out(self.hidden(tf.concat([id_vec, feats], axis=-1)))
    def call(self, item_ids):
        return self._forward(self.embedding(item_ids), tf.gather(self.item_feats, item_ids))
    def embed_cold(self, feats):
        # brand-new item: no learned ID vector yet -> use the mean ID vector + its content features
        table = tf.convert_to_tensor(self.embedding.embeddings)
        mean_id = tf.tile(tf.reduce_mean(table, axis=0, keepdims=True), [tf.shape(feats)[0], 1])
        return self._forward(mean_id, feats)

D = 32
query_tower = QueryTower(n_users, D, user_feats)
candidate_tower = CandidateTower(n_items, D, item_feats)
_ = query_tower(tf.constant([0])); _ = candidate_tower(tf.constant([0]))       # build weights

# logQ correction term: log of each item's probability of being sampled as an in-batch negative
freq = np.bincount(train_pos.i.values, minlength=n_items) + 1.0
log_q = tf.constant(np.log(freq / freq.sum()), tf.float32)

train_u = train_pos.u.values.astype("int32"); train_i = train_pos.i.values.astype("int32")
BATCH = 1024
ds = tf.data.Dataset.from_tensor_slices((train_u, train_i)).shuffle(len(train_u), seed=SEED, reshuffle_each_iteration=True).batch(BATCH, drop_remainder=True)
opt = tf.keras.optimizers.Adam(2e-3)

@tf.function
def train_step(u, i):
    with tf.GradientTape() as tape:
        q = query_tower(u); c = candidate_tower(i)
        logits = tf.matmul(q, c, transpose_b=True)                          # (B, B): row = user, col = candidate item
        logits -= tf.gather(log_q, i)[None, :]                              # logQ correction
        dup = tf.cast(tf.equal(i[:, None], i[None, :]), tf.float32) - tf.eye(tf.shape(i)[0])
        logits -= dup * 1e9                                                 # mask accidental hits
        labels = tf.range(tf.shape(i)[0])
        loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels, logits))
    v = query_tower.trainable_variables + candidate_tower.trainable_variables
    opt.apply_gradients(zip(tape.gradient(loss, v), v))
    return loss

def two_tower_exact_eval():
    Q = query_tower(tf.constant(np.arange(n_users), tf.int32)).numpy()
    Cm = candidate_tower(tf.constant(np.arange(n_items), tf.int32)).numpy()
    S = Q @ Cm.T; S[seen_mask] = -np.inf
    top = topk_from_scores(S, 100)
    return evaluate_recs({u: top[u] for u in eval_users}, eval_users)

EPOCHS = 30
for ep in range(1, EPOCHS + 1):
    losses = [float(train_step(u, i)) for u, i in ds]
    msg = f"epoch {ep:2d}  loss {np.mean(losses):.4f}"
    if ep % 10 == 0:
        msg += "  | test " + str({k: round(v, 4) for k, v in two_tower_exact_eval().items()})
    if ep % 5 == 0 or ep == 1 or ep % 10 == 0: print(msg)

In [ ]:
tt_metrics = two_tower_exact_eval()
RESULTS["two_tower_exact"] = tt_metrics
print("Two-tower (exact retrieval, seen items excluded):", {k: round(v, 4) for k, v in tt_metrics.items()})
print("Popularity baseline                            :", {k: round(v, 4) for k, v in base_pop.items()})

# ---- precompute the candidate-tower embedding for the WHOLE catalog and save it (Phase 3 deliverable) ----
C = candidate_tower(tf.constant(np.arange(n_items), tf.int32)).numpy().astype("float32")
np.save("item_embeddings.npy", C)
print("saved item_embeddings.npy", C.shape)

# ---- query tower runs ONLINE: wrap as a tf.function for fast single-request calls ----
@tf.function(input_signature=[tf.TensorSpec([None], tf.int32)])
def query_fn(u): return query_tower(u)
Qall = query_tower(tf.constant(np.arange(n_users), tf.int32)).numpy().astype("float32")

### Phase 2 revisited on the two-tower embeddings
(These play the role of the "Task 14 item embeddings" the readme asks for.)

In [ ]:
Cn = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-12)             # L2-normalised copy, used later for MMR similarity
print("Spearman(norm, popularity):", round(spearmanr(np.linalg.norm(C, axis=1)[pop > 0], pop[pop > 0])[0], 3))
tbl2, _ = comparison_table(C, q_idx)
print("Query:", titles[q_idx]); display(tbl2)
agg2 = {m: [] for m in ["dot", "cosine", "euclid"]}
for qi in qs:
    t = compare_measures(C, qi)
    for m in t: agg2[m].append(log_pop[t[m]].mean())
print(f"Mean log(1+pop) of top-10 over {len(qs)} queries:", {m: round(float(np.mean(v)), 2) for m, v in agg2.items()})

---
# Phase 4 — Approximate Nearest Neighbor (ANN) Search

**Why exact search doesn't scale.** Exact top-k means scoring the query against *every* item: `O(n·d)` per request. At `n = 10^7`, `d = 64` that's ~640 M multiply-adds per request, times thousands of requests per second.

**ANN's idea.** Build an index ahead of time (e.g. IVF: k-means clusters the items; at query time only scan the `nprobe` nearest clusters). You trade a little **recall** (you might miss a few true top-k items) for a large **speedup**.

> Note vs. the readme snippet: `IndexIVFFlat(quantizer, d, nlist)` defaults to **L2** distance. Since our retrieval score is an inner product, I pass `faiss.METRIC_INNER_PRODUCT` explicitly — otherwise the index would answer a different question than the one the model was trained on.

In [ ]:
d = C.shape[1]
K_RET = 100
bench_users = rng.choice(np.arange(n_users), 300, replace=False)
Qb = Qall[bench_users]

flat = faiss.IndexFlatIP(d); flat.add(C)                                # exact baseline
_, exact_ids = flat.search(Qb, K_RET)

nlist = 64
quantizer = faiss.IndexFlatIP(d)                                        # keep a reference so it isn't garbage-collected
ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
ivf.train(C); ivf.add(C)

def np_search(q):                                                       # brute-force NumPy scan
    s = C @ q[0]; idx = np.argpartition(-s, K_RET)[:K_RET]
    return idx[np.argsort(-s[idx])]

def ms_per_query(search_fn, Q, reps=1):
    t = time.perf_counter()
    for _ in range(reps):
        for q in Q: search_fn(q[None])
    return (time.perf_counter() - t) / (reps * len(Q)) * 1000

def recall_vs_exact(approx_ids, exact_ids):
    return float(np.mean([len(set(a[a >= 0]) & set(e)) / len(e) for a, e in zip(approx_ids, exact_ids)]))

np_ms   = ms_per_query(np_search, Qb)
flat_ms = ms_per_query(lambda q: flat.search(q, K_RET), Qb)
rows = [{"index": "NumPy brute force", "setting": "-", "ms/query": np_ms, "recall@100": 1.0},
        {"index": "FAISS IndexFlatIP (exact)", "setting": "-", "ms/query": flat_ms, "recall@100": 1.0}]
for nprobe in [1, 2, 4, 8, 16, 32, 64]:
    ivf.nprobe = nprobe
    ms = ms_per_query(lambda q: ivf.search(q, K_RET), Qb)
    _, ids = ivf.search(Qb, K_RET)
    rows.append({"index": "FAISS IVFFlat", "setting": f"nprobe={nprobe}/{nlist}", "ms/query": ms, "recall@100": recall_vs_exact(ids, exact_ids)})
ann_tbl = pd.DataFrame(rows); ann_tbl["ms/query"] = ann_tbl["ms/query"].round(3); ann_tbl["recall@100"] = ann_tbl["recall@100"].round(3)
display(ann_tbl)

ivf_rows = ann_tbl[ann_tbl["index"] == "FAISS IVFFlat"]
ok = ivf_rows[ivf_rows["recall@100"] >= 0.95]
chosen = (ok.iloc[0] if len(ok) else ivf_rows.iloc[-1])
CHOSEN_NPROBE = int(chosen.setting.split("=")[1].split("/")[0])
ivf.nprobe = CHOSEN_NPROBE
RESULTS.update(ivf_nprobe=CHOSEN_NPROBE, ivf_recall100=float(chosen["recall@100"]), ivf_ms=float(chosen["ms/query"]), flat_ms=flat_ms, np_ms=np_ms)
print(f"Chosen: nprobe={CHOSEN_NPROBE} -> recall@100={chosen['recall@100']}, {chosen['ms/query']} ms/query")

**The catalog here is tiny (9.7K items), so exact search is already fast.** The cell below repeats the experiment on a **synthetic 200K-item catalog** (clustered vectors, so ANN has structure to exploit like real embeddings do) to show where ANN actually pays off.

In [ ]:
rs = np.random.default_rng(0); N = 200_000
centers = rs.standard_normal((300, d)).astype("float32") * 3
mk = lambda n: (centers[rs.integers(0, 300, n)] + rs.standard_normal((n, d))).astype("float32")
big, bq = mk(N), mk(100)

big_flat = faiss.IndexFlatIP(d); big_flat.add(big)
_, big_exact = big_flat.search(bq, K_RET)
bq_ivf_q = faiss.IndexFlatIP(d)
big_ivf = faiss.IndexIVFFlat(bq_ivf_q, d, 512, faiss.METRIC_INNER_PRODUCT); big_ivf.train(big); big_ivf.add(big)

def big_np(q):
    s = big @ q[0]; idx = np.argpartition(-s, K_RET)[:K_RET]; return idx[np.argsort(-s[idx])]
big_rows = [{"method": "NumPy brute force", "ms/query": ms_per_query(big_np, bq), "recall@100": 1.0},
            {"method": "FAISS Flat (exact)", "ms/query": ms_per_query(lambda q: big_flat.search(q, K_RET), bq), "recall@100": 1.0}]
for npb in [4, 16, 32]:
    big_ivf.nprobe = npb
    _, ids = big_ivf.search(bq, K_RET)
    big_rows.append({"method": f"FAISS IVF nprobe={npb}/512", "ms/query": ms_per_query(lambda q: big_ivf.search(q, K_RET), bq), "recall@100": recall_vs_exact(ids, big_exact)})
big_tbl = pd.DataFrame(big_rows); big_tbl["ms/query"] = big_tbl["ms/query"].round(3); big_tbl["recall@100"] = big_tbl["recall@100"].round(3)
display(big_tbl)
sp = big_tbl.iloc[0]["ms/query"] / big_tbl.iloc[3]["ms/query"]                 # NumPy vs IVF(nprobe=16)
print(f"At 200K items, IVF(nprobe=16) is ~{sp:.0f}x faster than the NumPy scan. The gap grows linearly with catalog size.")
del big, big_flat, big_ivf

**Tradeoff statement (Phase 4 deliverable).** I picked the smallest `nprobe` that keeps **recall@100 >= 0.95** against exact search (value printed above). Going lower saves latency but starts dropping genuinely good candidates — and since a candidate that retrieval misses can *never* be rescued by the scoring stage, retrieval recall puts a ceiling on final quality. Going higher buys back the last few percent of recall at near-linear latency cost. At this classroom catalog size the absolute savings are small; at 10^7 items they decide whether the system can serve at all.

**Why this makes Phase 1 possible.** Retrieval has to look at the *entire* catalog on every request inside a ~10 ms slice of the budget. Only an index structure that avoids touching most of the items can do that.

Below is the retrieval helper the rest of the notebook uses. Because users must never be shown things they've already seen, it **over-fetches** (`K + number_seen`) and then filters; if the ANN probe doesn't return enough unseen items, it widens `nprobe` and retries. (Production systems usually push this filter into the index via an ID-selector or a Bloom filter; over-fetch is the simple portable version.)

In [ ]:
pipe_index = ivf          # the index served in the pipeline

def retrieve(u, q, K=300, index=None):
    index = index or pipe_index
    seen = seen_mask[u]
    k_fetch = int(min(n_items, K + int(n_seen[u])))
    orig = getattr(index, "nprobe", None)
    try:
        while True:
            Dv, Iv = index.search(q, k_fetch)
            Iv, Dv = Iv[0], Dv[0]
            ok = Iv >= 0
            Iv, Dv = Iv[ok], Dv[ok]
            keep = ~seen[Iv]
            Iv, Dv = Iv[keep], Dv[keep]
            if len(Iv) >= K or orig is None or index.nprobe >= index.nlist: break
            index.nprobe = min(index.nlist, index.nprobe * 2)           # widen the probe and retry
    finally:
        if orig is not None: index.nprobe = orig
    return Iv[:K], Dv[:K]

# quick check: two-tower + ANN retrieval quality, measured on held-out test items
def retrieval_recall(users, K, index=None):
    return float(np.mean([len(set(retrieve(u, Qall[u][None], K, index)[0]) & test_items[u]) / len(test_items[u]) for u in users]))
for K in [100, 300]:
    print(f"ANN retrieval  test-item recall@{K}: {retrieval_recall(sample_users, K):.4f}")

---
# Phase 5 — Scoring: The Ranking Model

The scoring stage re-ranks the shortlist with a model that is **too expensive to run on the whole catalog but affordable on ~300 items**. It can use **cross-features**: because user and item representations interact *inside* the network, it can learn things like "this user likes sci-fi *only when* it's recent" that two independent towers cannot express.

Stand-in for the Task 14 model: **NeuMF** = GMF branch (element-wise product of embeddings) + MLP branch (concatenated embeddings through dense layers) -> sigmoid. Trained with binary cross-entropy against 4 sampled negatives per positive (resampled each epoch).

In [ ]:
def build_neumf(n_users, n_items, mf_dim=16, mlp_dim=32):
    u_in = tf.keras.Input(shape=(), dtype="int32"); i_in = tf.keras.Input(shape=(), dtype="int32")
    Emb = tf.keras.layers.Embedding
    gmf = tf.keras.layers.Multiply()([Emb(n_users, mf_dim)(u_in), Emb(n_items, mf_dim)(i_in)])
    mlp = tf.keras.layers.Concatenate()([Emb(n_users, mlp_dim)(u_in), Emb(n_items, mlp_dim)(i_in)])
    for units in (64, 32, 16): mlp = tf.keras.layers.Dense(units, activation="relu")(mlp)
    out = tf.keras.layers.Dense(1, activation="sigmoid")(tf.keras.layers.Concatenate()([gmf, mlp]))
    return tf.keras.Model([u_in, i_in], out)                              # output shape (batch, 1)

neumf = build_neumf(n_users, n_items)
neumf.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy")

def make_epoch_data(n_neg=4):
    u = np.repeat(train_u, n_neg); neg = rng.integers(0, n_items, len(u))
    bad = seen_mask[u, neg]
    while bad.any():
        neg[bad] = rng.integers(0, n_items, bad.sum()); bad = seen_mask[u, neg]
    U = np.concatenate([train_u, u]); I = np.concatenate([train_i, neg]).astype("int32")
    Y = np.concatenate([np.ones(len(train_u)), np.zeros(len(u))]).astype("float32")
    p = rng.permutation(len(U)); return U[p].astype("int32"), I[p], Y[p].reshape(-1, 1)

for ep in range(8):
    U, I, Y = make_epoch_data()
    h = neumf.fit([U, I], Y, batch_size=2048, epochs=1, verbose=0)
    print(f"NeuMF epoch {ep + 1}  loss {h.history['loss'][0]:.4f}")

In [ ]:
@tf.function(input_signature=[tf.TensorSpec([None], tf.int32), tf.TensorSpec([None], tf.int32)])
def neumf_score(u, i): return tf.squeeze(neumf([u, i], training=False), -1)

# ---- brute-force baseline: score the ENTIRE catalog for every user (feasible only because this catalog is tiny) ----
t0 = time.time()
uu = np.repeat(np.arange(n_users, dtype="int32"), n_items); ii = np.tile(np.arange(n_items, dtype="int32"), n_users)
S_neumf = neumf.predict([uu, ii], batch_size=65536, verbose=0).reshape(n_users, n_items).astype("float32")
del uu, ii
print(f"full-catalog NeuMF scoring for all users took {time.time() - t0:.1f}s")

S_mask = S_neumf.copy(); S_mask[seen_mask] = -np.inf
bf_top = topk_from_scores(S_mask, 100)
bf_metrics = evaluate_recs({u: bf_top[u] for u in eval_users}, eval_users)
RESULTS["neumf_bruteforce"] = bf_metrics
print("NeuMF brute force over whole catalog:", {k: round(v, 4) for k, v in bf_metrics.items()})

In [ ]:
def two_stage_topn(u, K=300, n=10):
    cand, _ = retrieve(u, Qall[u][None], K)
    sc = neumf_score(tf.constant(np.full(len(cand), u, np.int32)), tf.constant(cand.astype(np.int32))).numpy()
    o = np.argsort(-sc)
    return cand[o][:n], cand

def two_stage_eval(users, K):
    ov, cr, nd_ts, nd_bf = [], [], [], []
    for u in users:
        top, cand = two_stage_topn(u, K)
        bf10 = bf_top[u][:10]
        ov.append(len(set(top) & set(bf10)) / 10)                 # overlap@10 with brute force
        cr.append(len(set(bf10) & set(cand)) / 10)                 # did retrieval even hand over the true best 10?
        nd_ts.append(ndcg_at_k(top, test_items[u])); nd_bf.append(ndcg_at_k(bf10, test_items[u]))
    return dict(overlap10=np.mean(ov), cand_recall_of_bf_top10=np.mean(cr), ndcg10_two_stage=np.mean(nd_ts), ndcg10_bruteforce=np.mean(nd_bf))

rows = []
for K in [50, 100, 300, 500]:
    r = two_stage_eval(sample_users, K); rows.append({"K candidates": K, **{k: round(v, 4) for k, v in r.items()}})
    if K == 300: RESULTS["two_stage_K300"] = r
display(pd.DataFrame(rows))

In [ ]:
# side-by-side for one sample user
u0 = int(sample_users[0]); top_ts, cand0 = two_stage_topn(u0, 300)
cmp = pd.DataFrame({"rank": range(1, 11),
                    "retrieve-then-score": titles[top_ts], "brute-force NeuMF": titles[bf_top[u0][:10]]})
print(f"user {u0}: overlap@10 = {len(set(top_ts) & set(bf_top[u0][:10]))}/10 (candidate set size {len(cand0)} of {n_items} items = {len(cand0) / n_items:.1%} of the catalog)")
display(cmp)

**Practical lesson (Phase 5 deliverable).** The two-stage pipeline scores only a few percent of the catalog yet its top-10 closely matches full-catalog scoring (see `overlap10` and `cand_recall_of_bf_top10` above — they rise with K and saturate). The expensive scorer is only affordable *because* retrieval is good, and retrieval only has to be good in one specific sense: **its candidate set must contain the items the scorer would have ranked at the top.** When that recall drops, the final list degrades no matter how strong the scorer is.

---
# Phase 6 — Re-Ranking: Diversity, Freshness, Fairness & Business Rules

Re-ranking takes the **scored candidates** and applies list-level goals. Here it does five things:

1. **Exclude seen items** (business rule — already enforced by `retrieve`, quantified below).
2. **Diversity via MMR**: `score'(i) = lam * relevance(i) - (1-lam) * max_{j in selected} sim(i, j)`, similarity = cosine of candidate-tower embeddings.
3. **Freshness boost**: `+ fresh_w * freshness(i)`, where freshness decays with release year.
4. **Business rule — per-genre cap**: at most `max_per_genre` items sharing a primary genre. (MovieLens has no creator/brand field, so this is the stand-in for the readme's "don't show two items from the same creator back-to-back"; with creator data you'd cap on creator ID in exactly the same way.)
5. **Fairness — tail exposure quota**: guarantee at least `min_tail` items from outside the popularity head (top-20% of items). A pure relevance scorer systematically starves anything without engagement history; this is the simplest enforcement mechanism.

In [ ]:
def rerank(cand, rel, top_n=10, lam=0.7, fresh_w=0.1, max_per_genre=None, min_tail=0, use_mmr=True):
    cand = np.asarray(cand); rel = np.asarray(rel, float)
    r = (rel - rel.min()) / (rel.max() - rel.min() + 1e-9)               # min-max relevance within the candidate set
    base = r + fresh_w * freshness[cand]
    E = Cn[cand]; sim = E @ E.T
    pg = primary_genre[cand]
    selected, gcount = [], {}
    maxsim = np.zeros(len(cand)); taken = np.zeros(len(cand), bool)
    while len(selected) < min(top_n, len(cand)):
        score = (lam * base - (1 - lam) * maxsim) if use_mmr else base.copy()
        score[taken] = -np.inf
        if max_per_genre is not None:
            capped = [g for g, c in gcount.items() if c >= max_per_genre and g >= 0]
            if capped:
                blocked = np.isin(pg, capped) & ~taken
                if (~blocked & ~taken).any(): score[blocked] = -np.inf   # relax the cap only if it would block everything
        j = int(np.argmax(score))
        selected.append(j); taken[j] = True
        maxsim = np.maximum(maxsim, sim[:, j]); gcount[pg[j]] = gcount.get(pg[j], 0) + 1
    if min_tail > 0:
        tail_c = is_tail[cand]; n_tail = sum(tail_c[j] for j in selected)
        if n_tail < min_tail:
            pool = [j for j in np.argsort(-base) if tail_c[j] and j not in selected]
            for j in pool[: min_tail - n_tail]:
                droppable = [x for x in selected if not tail_c[x]]
                if not droppable: break
                drop = min(droppable, key=lambda x: base[x]); selected[selected.index(drop)] = j
    return cand[selected]

def ild(ids):
    E = Cn[np.asarray(ids)]; S = E @ E.T; n = len(ids)
    return float((1 - S)[~np.eye(n, dtype=bool)].mean())

# cache scored candidates (K=300) for the sample users: used for all ablations below
cache = {}
for u in sample_users:
    cand, _ = retrieve(u, Qall[u][None], 300)
    sc = neumf_score(tf.constant(np.full(len(cand), u, np.int32)), tf.constant(cand.astype(np.int32))).numpy()
    cache[int(u)] = (cand, sc)
print("cached scored candidates for", len(cache), "users")

In [ ]:
# ---- business rule #1: how much of an un-filtered recommender's top-10 has the user ALREADY seen? ----
raw_unfiltered = topk_from_scores(S_neumf[sample_users], 10)
frac_seen = np.mean([seen_mask[u, raw_unfiltered[k]].mean() for k, u in enumerate(sample_users)])
print(f"Without the 'exclude seen' rule, {frac_seen:.1%} of NeuMF's top-10 are items the user already interacted with.")

In [ ]:
CONFIGS = [
    ("1. raw scored (seen excluded)",   dict(use_mmr=False, fresh_w=0.0)),
    ("2. + MMR diversity (lam=0.7)",    dict(lam=0.7, fresh_w=0.0)),
    ("3. + freshness boost (0.1)",      dict(lam=0.7, fresh_w=0.1)),
    ("4. + genre cap (max 3)",          dict(lam=0.7, fresh_w=0.1, max_per_genre=3)),
    ("5. + tail quota (>=2 tail)",      dict(lam=0.7, fresh_w=0.1, max_per_genre=3, min_tail=2)),
]
def summarize(recs):
    us = list(recs)
    return {"NDCG@10": np.mean([ndcg_at_k(recs[u], test_items[u]) for u in us]),
            "Hit@10": np.mean([hit_at_k(recs[u], test_items[u]) for u in us]),
            "ILD@10 (diversity)": np.mean([ild(recs[u]) for u in us]),
            "mean release year": np.mean([item_year[recs[u]].mean() for u in us]),
            "tail share": np.mean([is_tail[recs[u]].mean() for u in us]),
            "mean log-pop": np.mean([log_pop[recs[u]].mean() for u in us])}
rows = []
for name, kw in CONFIGS:
    recs = {u: rerank(c, s, **kw) for u, (c, s) in cache.items()}
    rows.append({"re-ranking stack": name, **summarize(recs)})
rr_tbl = pd.DataFrame(rows).set_index("re-ranking stack").round(4)
RESULTS["rerank_table"] = rr_tbl
display(rr_tbl)

In [ ]:
# lambda sweep: the relevance <-> diversity dial
sweep = []
for lam in [1.0, 0.9, 0.8, 0.7, 0.5, 0.3]:
    recs = {u: rerank(c, s, lam=lam, fresh_w=0.0) for u, (c, s) in cache.items()}
    sm = summarize(recs); sweep.append((lam, sm["NDCG@10"], sm["ILD@10 (diversity)"]))
sw = pd.DataFrame(sweep, columns=["lambda", "NDCG@10", "ILD@10"]).round(4); display(sw)
fig, ax = plt.subplots(figsize=(5.5, 4)); ax.plot(sw["ILD@10"], sw["NDCG@10"], "o-")
for _, r in sw.iterrows(): ax.annotate(f"lam={r['lambda']}", (r["ILD@10"], r["NDCG@10"]), fontsize=8)
ax.set_xlabel("diversity (ILD@10)"); ax.set_ylabel("relevance (NDCG@10)"); ax.set_title("Relevance vs diversity (MMR lambda)"); plt.show()

In [ ]:
# qualitative before/after for one user
u1 = int(sample_users[1]); c1, s1 = cache[u1]
raw10 = c1[np.argsort(-s1)[:10]]; fin10 = rerank(c1, s1, **CONFIGS[-1][1])
view = lambda ids: [f"{titles[i]} [{genres_str[i].split('|')[0]}{', tail' if is_tail[i] else ''}]" for i in ids]
display(pd.DataFrame({"raw scored top-10": view(raw10), "re-ranked top-10": view(fin10)}, index=range(1, 11)))

**Tradeoff discussion (Phase 6 deliverable).** Read the table top to bottom — each row adds one goal on top of the previous:

- **MMR** raises ILD (lists are less repetitive) and typically costs a small amount of NDCG; the lambda sweep makes the exchange rate explicit — pick lam where the diversity gain is large relative to the NDCG you give up (the knee of the curve).
- **Freshness** shifts the mean release year upward; because MovieLens test positives aren't biased toward new releases, it usually costs a little offline NDCG. In a real product where recency drives engagement, the *online* effect can be the opposite — that's what A/B tests are for.
- **Genre cap** and **tail quota** are constraints rather than scores: they mostly move diversity and tail share, and cost relevance only to the extent the displaced items were genuinely top-ranked.
- Offline NDCG alone will always look like "the raw list wins" since the raw list is optimised for exactly that. The right question is whether the *combined* online outcome (engagement, retention, creator ecosystem health) improves — so the re-ranker's parameters (`lam`, `fresh_w`, caps, quota) are product knobs to tune with experiments, not model hyper-parameters to tune offline.
- **Fairness** note: the tail quota here is an exposure floor. It's a blunt instrument; real systems often use exposure-aware re-ranking (e.g. proportional exposure by creator group) and monitor outcome metrics per group.

---
# Phase 7 — Serving Infrastructure & Latency Budgets

### 7.1 Offline vs online
| Component | When it runs | Why |
|---|---|---|
| Candidate-tower embeddings + ANN index | **Offline** — rebuilt nightly/hourly | They depend only on items, not on the request; precomputing makes retrieval a lookup |
| Query tower | **Online**, once per request | The user's context (recent activity, time of day) changes between requests; one tiny forward pass |
| ANN search -> scoring -> re-ranking | **Online** | Per-request, latency-critical |

### 7.2 Staleness tradeoff
A brand-new item isn't in the index until the next rebuild, so **retrieval can't return it**, no matter how good it is (Exercise 4 traces this concretely). Mitigations: more frequent index refreshes, streaming adds to the index (`add_with_ids`), and a separate content-based / "new & trending" candidate source that bypasses the stale index. This is the item cold-start problem from Task 13, now seen as an infrastructure constraint.

### 7.3 The full pipeline function, instrumented with per-stage timers

In [ ]:
RERANK_CFG = dict(lam=0.7, fresh_w=0.1, max_per_genre=3, min_tail=2)

def recommend_pipeline(u, top_n=10, K=300, timings=None, extra_cands=None, do_rerank=True, index=None, rr=None):
    t0 = time.perf_counter()
    q = query_fn(tf.constant([u], tf.int32)).numpy().astype("float32")                    # ONLINE query tower
    t1 = time.perf_counter()
    cand, _ = retrieve(u, q, K, index)                                                    # stage 1: ANN retrieval
    if extra_cands is not None:
        extra = np.asarray(extra_cands); extra = extra[~seen_mask[u][extra]]
        cand = np.unique(np.concatenate([cand, extra]))
    t2 = time.perf_counter()
    sc = neumf_score(tf.constant(np.full(len(cand), u, np.int32)), tf.constant(cand.astype(np.int32))).numpy()   # stage 2
    t3 = time.perf_counter()
    final = rerank(cand, sc, top_n=top_n, **(rr or RERANK_CFG)) if do_rerank else cand[np.argsort(-sc)[:top_n]]    # stage 3
    t4 = time.perf_counter()
    if timings is not None:
        timings.update(query_tower=(t1 - t0) * 1e3, ann_retrieval=(t2 - t1) * 1e3, scoring=(t3 - t2) * 1e3, reranking=(t4 - t3) * 1e3, total=(t4 - t0) * 1e3)
    return final, cand, sc

# warm-up (first calls trace the tf.functions), then measure many requests
for u in sample_users[:20]: recommend_pipeline(int(u))
lat = []
for u in rng.choice(eval_users, 300):
    tm = {}; recommend_pipeline(int(u), timings=tm); lat.append(tm)
lat = pd.DataFrame(lat)

BUDGET = {"query_tower": 5, "ann_retrieval": 10, "scoring": 60, "reranking": 10}
BUDGET["overhead_network"] = 100 - sum(BUDGET.values())
BUDGET["total"] = 100
meas = lat.describe(percentiles=[.5, .95, .99]).T[["mean", "50%", "95%", "99%"]].round(2)
meas.columns = ["mean_ms", "p50_ms", "p95_ms", "p99_ms"]
budget_tbl = pd.DataFrame({"budget_ms": pd.Series(BUDGET)}).join(meas, how="left")
display(budget_tbl.loc[["query_tower", "ann_retrieval", "scoring", "reranking", "overhead_network", "total"]])

In [ ]:
stages = ["query_tower", "ann_retrieval", "scoring", "reranking"]
p50 = lat[stages].median(); bott = p50.idxmax()
RESULTS.update(lat_p50=float(lat.total.median()), lat_p95=float(lat.total.quantile(.95)), lat_p99=float(lat.total.quantile(.99)), bottleneck=bott, stage_p50=p50.to_dict())
fig, ax = plt.subplots(figsize=(7, 1.8))
left = 0
for s in stages:
    ax.barh(0, p50[s], left=left, label=f"{s} ({p50[s]:.1f} ms)"); left += p50[s]
ax.axvline(100, color="red", ls="--", label="100 ms budget"); ax.set_yticks([]); ax.set_xlabel("ms (p50)")
ax.legend(ncol=3, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.5)); plt.title("Measured p50 latency by stage"); plt.show()
print(f"Measured end-to-end: p50={RESULTS['lat_p50']:.1f} ms, p95={RESULTS['lat_p95']:.1f} ms, p99={RESULTS['lat_p99']:.1f} ms")
print(f"Biggest bottleneck (p50): {bott} at {p50[bott]:.1f} ms ({p50[bott] / p50.sum():.0%} of pipeline time)")

In [ ]:
# proposed fix: shrink the candidate set handed to scoring, and measure what it costs in quality
def pipeline_profile(K, users, do_rerank=False):
    tms, nd = [], []
    for u in users:
        tm = {}; fin, cand, _ = recommend_pipeline(int(u), K=K, timings=tm, do_rerank=do_rerank); tms.append(tm); nd.append(ndcg_at_k(fin, test_items[u]))
    t = pd.DataFrame(tms); return t.total.median(), t[["ann_retrieval", "scoring"]].median(), float(np.mean(nd))
rows = []
for K in [50, 100, 300, 500]:
    tot, parts, nd = pipeline_profile(K, sample_users[:100])
    rows.append({"K": K, "p50 total ms": round(tot, 2), "ann ms": round(parts.ann_retrieval, 2), "scoring ms": round(parts.scoring, 2), "NDCG@10 (no re-rank)": round(nd, 4)})
display(pd.DataFrame(rows))

**Bottleneck analysis (Phase 7 deliverable).** The cell above prints the measured bottleneck for *your* machine. In typical runs it is either `scoring` (per-request cost grows with candidate count and the model is the heaviest component) or `ann_retrieval` for heavy users (over-fetching `K + n_seen` to filter seen items).

**Proposed fixes, in order of effort:**
1. **Shrink K** (table above): scoring cost falls roughly linearly with candidate count; pick the smallest K where NDCG has plateaued.
2. **Cheaper scorer / distillation:** distil NeuMF into a smaller MLP, or drop to a 2-layer cross model, for the same candidates.
3. **Push the seen-filter into the index** (ID selector / Bloom filter) so retrieval no longer needs to over-fetch.
4. **Batch / quantise** the scorer (fp16/int8, batched requests) once it is served from a dedicated model server.

Note also why the **budget is stated in percentiles, not averages:** users feel p95/p99, and tail latency is dominated by the heavy-history users above.

---
# Phase 8 — Capstone: Full Pipeline, Monitoring & the A/B Loop

### 8.1 `get_recommendations(user_id, top_n=10)`

In [ ]:
def get_recommendations(user_id, top_n=10, K=300):
    # Full pipeline: query tower -> ANN retrieval (unseen only) -> NeuMF scoring -> re-ranking
    final, cand, sc = recommend_pipeline(int(user_id), top_n=top_n, K=K)
    smap = dict(zip(cand.tolist(), sc.tolist()))
    return pd.DataFrame({"rank": range(1, len(final) + 1), "title": titles[final], "year": item_year[final].astype(int),
                         "genres": genres_str[final], "score": [round(smap[int(i)], 3) for i in final],
                         "train_pop": pop[final], "tail": is_tail[final]})

# pick 3 users with different dominant tastes for the qualitative check
picked, seen_top = [], set()
for u in sample_users:
    g = int(np.argmax(user_genre[u]))
    if g not in seen_top and cnt[u] >= 30:
        picked.append(int(u)); seen_top.add(g)
    if len(picked) == 3: break

for u in picked:
    tg = [all_genres[g] for g in np.argsort(-user_genre[u])[:3]]
    liked = train_pos[train_pos.u == u].sort_values("timestamp").tail(5).i.values
    print("=" * 100)
    print(f"USER {u} | top genres in history: {tg} | {cnt[u]} train positives")
    print("recently liked:", [titles[i] for i in liked])
    recs = get_recommendations(u); display(recs)
    match = np.mean([any(g in gs for g in tg) for gs in recs.genres])
    print(f"-> {match:.0%} of recommendations share a genre with the user's top-3 genres")

### 8.2 End-to-end comparison of every stage

In [ ]:
# the final pipeline for all eval users, compared against each simpler system
final_recs = {int(u): recommend_pipeline(int(u))[0] for u in eval_users}
S_tt = Qall @ C.T; S_tt[seen_mask] = -np.inf; tt_top = topk_from_scores(S_tt, 10)
ts_recs = {int(u): two_stage_topn(int(u), 300)[0] for u in eval_users}
systems = {
    "Popularity baseline": {int(u): pop_rank[u][:10] for u in eval_users},
    "Two-tower only (exact)": {int(u): tt_top[u] for u in eval_users},
    "NeuMF brute force (whole catalog)": {int(u): bf_top[u][:10] for u in eval_users},
    "Retrieve + score (K=300)": ts_recs,
    "Full pipeline (+ re-rank)": final_recs,
}
final_tbl = pd.DataFrame({k: summarize(v) for k, v in systems.items()}).T.round(4)
RESULTS["final_table"] = final_tbl
display(final_tbl)

### 8.3 Production monitoring
Signals to track continuously, with the alert idea for each:

| Signal | How | Alert when |
|---|---|---|
| **Offline metric drift** | NDCG@10 / MAP (Task 13) recomputed on a rolling window of fresh interactions | drops > X% vs. trailing baseline |
| **Online engagement** | CTR, watch-time, session length, 7-day retention, by experiment arm | guardrail metrics regress |
| **Latency percentiles** | p50 / p95 / p99 per stage (not just the mean) | p95 or p99 exceeds the stage budget |
| **Retrieval <-> scoring consistency** | share of the scorer's would-be top-10 (over a bigger pool) that retrieval actually surfaced | falls below ~0.9 -> retrieval and scoring have drifted apart (e.g. stale index, new scorer features) |
| **Catalog health** | coverage, tail-exposure share, share of items added in last N days that ever got retrieved | new items stay at zero impressions (index staleness) |
| **Data/feature health** | missing-feature rates, embedding-norm distribution, index age | feature pipeline break or index not rebuilt |

The cell below is a small working version of the first four signals.

In [ ]:
def monitor_report(users, recs, K=300, ndcg_alert_drop=0.15, consistency_alert=0.9):
    users = [int(u) for u in users]
    med_ts = test_pos.timestamp.median()                                  # two time windows stand in for a rolling window
    tsmap = test_pos.set_index(["u", "i"]).timestamp.to_dict()
    A = {u: {i for i in test_items[u] if tsmap.get((u, i), 0) < med_ts} for u in users}
    B = {u: {i for i in test_items[u] if tsmap.get((u, i), 0) >= med_ts} for u in users}
    ndA = np.mean([ndcg_at_k(recs[u], A[u]) for u in users if A[u]]); ndB = np.mean([ndcg_at_k(recs[u], B[u]) for u in users if B[u]])
    cons = np.mean([len(set(bf_top[u][:10]) & set(retrieve(u, Qall[u][None], K)[0])) / 10 for u in users])
    return {"NDCG@10 window A": round(ndA, 4), "NDCG@10 window B": round(ndB, 4),
            "NDCG relative change A->B": f"{(ndB - ndA) / max(ndA, 1e-9):+.1%}", "ALERT ndcg drift": bool((ndA - ndB) / max(ndA, 1e-9) > ndcg_alert_drop),
            "retrieval/scoring consistency": round(float(cons), 4), "ALERT consistency": bool(cons < consistency_alert),
            "latency p50/p95/p99 ms": (round(RESULTS["lat_p50"], 1), round(RESULTS["lat_p95"], 1), round(RESULTS["lat_p99"], 1)),
            "tail share in recs": round(float(np.mean([is_tail[recs[u]].mean() for u in users])), 3)}
mon = monitor_report(sample_users, final_recs); RESULTS["monitor"] = mon
for k, v in mon.items(): print(f"{k:34s} {v}")

### 8.4 The A/B feedback loop (building on Task 13, Phase 8)
**Roll out the stages independently, not just as one bundle:**

- **Retrieval model change** (new two-tower, new index type, new candidate source). *Primary metric:* candidate-generation **recall** offline, then online engagement. *Why separately:* a retrieval change alters *which items are even eligible*, so its effect can be hidden or amplified by the scorer; and retrieval regressions (missed items) are invisible to the scorer — it can't rank what it never sees.
- **Scoring model change.** *Primary metric:* online CTR / watch-time (offline NDCG as a gate). *Why separately:* it changes ordering within a fixed candidate set, and its bias risks (popularity feedback loops) are different.
- **Re-ranking rule change.** *Primary metric:* the *product* metric the rule exists for (e.g. long-term retention, creator exposure), with short-term engagement as a **guardrail**, because these rules intentionally trade short-term relevance.

Testing only the combined pipeline makes any movement un-attributable: if CTR drops, you don't know whether retrieval lost recall, the scorer got worse, or a re-ranking rule bit. Independent tests also allow different cadences — rankers retrain weekly, indexes rebuild hourly, rules change whenever product decides.

**Sample size for a CTR test** (how long the A/B must run):

In [ ]:
from scipy.stats import norm
def n_per_arm(p0, rel_lift, alpha=0.05, power=0.8):
    p1 = p0 * (1 + rel_lift); za, zb = norm.ppf(1 - alpha / 2), norm.ppf(power); pb = (p0 + p1) / 2
    return int(np.ceil(((za * np.sqrt(2 * pb * (1 - pb)) + zb * np.sqrt(p0 * (1 - p0) + p1 * (1 - p1))) ** 2) / (p1 - p0) ** 2))
pd.DataFrame([{"baseline CTR": p0, "relative lift to detect": f"{l:.0%}", "users per arm": n_per_arm(p0, l)} for p0 in (0.05, 0.10) for l in (0.01, 0.02, 0.05)])

### 8.5 One-page production readiness summary (auto-filled from this run's measurements)

In [ ]:
r = RESULTS; ts = r["two_stage_K300"]; rr = r["rerank_table"]
summary = f'''
## Production Readiness Summary — MovieLens Recommender

**1. Architecture (Phase 1).** Three stages: *candidate generation* (two-tower + FAISS IVF) -> *scoring* (NeuMF with cross-features) -> *re-ranking* (MMR diversity, freshness, genre cap, tail-exposure quota, seen-item exclusion). The query tower runs online; item embeddings and the ANN index are built offline.

**2. Retrieval & its recall tradeoff (Phases 2-4).** Item embeddings have dimension {D}. Dot product is norm-sensitive (popularity bias), so retrieval uses the dot product the model was *trained* with and relies on re-ranking to correct for bias. The IVF index is served at `nprobe={r["ivf_nprobe"]}/{nlist}`, giving **recall@100 = {r["ivf_recall100"]:.3f}** vs. exact search at {r["ivf_ms"]:.3f} ms/query (exact FAISS {r["flat_ms"]:.3f} ms, NumPy {r["np_ms"]:.3f} ms). On the 200K synthetic catalog the speedup over a NumPy scan was ~{sp:.0f}x.

**3. Scoring (Phase 5).** NeuMF over K=300 retrieved candidates reproduces the brute-force full-catalog top-10 with **overlap@10 = {ts["overlap10"]:.2f}** and candidate recall of the true top-10 = **{ts["cand_recall_of_bf_top10"]:.2f}**; NDCG@10 {ts["ndcg10_two_stage"]:.4f} vs {ts["ndcg10_bruteforce"]:.4f} for brute force — while scoring only {300 / n_items:.1%} of the catalog.

**4. Re-ranking (Phase 6).** Final stack (lam=0.7, fresh_w=0.1, max 3/genre, >=2 tail items): NDCG@10 {rr.iloc[-1]["NDCG@10"]:.4f} (raw scored: {rr.iloc[0]["NDCG@10"]:.4f}), ILD@10 {rr.iloc[-1]["ILD@10 (diversity)"]:.3f} (raw: {rr.iloc[0]["ILD@10 (diversity)"]:.3f}), tail share {rr.iloc[-1]["tail share"]:.0%} (raw: {rr.iloc[0]["tail share"]:.0%}). We accept a small offline relevance cost for diversity and exposure; the online A/B decides whether it pays off.

**5. Latency (Phase 7).** Measured end-to-end p50 **{r["lat_p50"]:.1f} ms**, p95 {r["lat_p95"]:.1f} ms, p99 {r["lat_p99"]:.1f} ms vs. a 100 ms budget. Biggest stage: **{r["bottleneck"]}** ({r["stage_p50"][r["bottleneck"]]:.1f} ms p50). Fix: reduce K / cheaper scorer / index-side seen-filter.

**6. Monitoring & A/B plan (Phase 8).** Track rolling NDCG, CTR/retention, per-stage p50/p95/p99, retrieval<->scoring consistency (now {r["monitor"]["retrieval/scoring consistency"]}), and new-item exposure. Roll out retrieval, scoring and re-ranking changes as **separate experiments** with stage-specific primary metrics and shared engagement guardrails.

**Known risks / next steps.** Index staleness for brand-new items (needs streaming adds + a content-based source — see Exercise 4); popularity feedback loops from training on logged exposures; offline NDCG on a small dataset is noisy, so treat the numbers as directional.
'''
display(Markdown(summary))

---
# Exercises (Must Do)

## Exercise 1 — `IndexIVFFlat` vs `IndexHNSWFlat`

In [ ]:
hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 200
t0 = time.time(); hnsw.add(C); build_s = time.time() - t0
rows = []
for ef in [8, 16, 32, 64, 128]:
    hnsw.hnsw.efSearch = ef
    ms = ms_per_query(lambda q: hnsw.search(q, K_RET), Qb); _, ids = hnsw.search(Qb, K_RET)
    rows.append({"index": "HNSW", "setting": f"efSearch={ef}", "ms/query": round(ms, 3), "recall@100": round(recall_vs_exact(ids, exact_ids), 3)})
hn = pd.DataFrame(rows)
ivf_cmp = ann_tbl[ann_tbl["index"] == "FAISS IVFFlat"][["index", "setting", "ms/query", "recall@100"]]
display(pd.concat([ivf_cmp, hn], ignore_index=True))
print(f"HNSW build time: {build_s:.2f}s")

**Takeaway.** IVF is cheap to build, has a simple recall/latency dial (`nprobe`), and is memory-light (with PQ variants it compresses further); HNSW (graph-based) usually reaches higher recall at a given latency on larger catalogs but costs more to build and more memory, and supports incremental adds well. HNSW with inner-product has no formal metric guarantee, so always verify recall against exact search as done here. At 9.7K items both are already fast, so differences show mostly in recall-at-fixed-latency, not absolute speed.

## Exercise 2 — Candidate set size 100 -> 500: NDCG@10 vs latency

In [ ]:
rows = []
for K in [50, 100, 200, 300, 500]:
    tms, nd, tr = [], [], []
    for u in sample_users:
        tm = {}; fin, cand, _ = recommend_pipeline(int(u), K=K, timings=tm, do_rerank=False)
        tms.append(tm["total"]); nd.append(ndcg_at_k(fin, test_items[u])); tr.append(len(set(cand) & test_items[u]) / len(test_items[u]))
    rows.append({"K": K, "NDCG@10 (retrieve+score)": round(np.mean(nd), 4), "test-item recall in candidate set": round(np.mean(tr), 4), "p50 latency ms": round(np.median(tms), 2)})
ex2 = pd.DataFrame(rows); display(ex2)
fig, ax = plt.subplots(figsize=(6, 3.6)); ax.plot(ex2.K, ex2["NDCG@10 (retrieve+score)"], "o-", label="NDCG@10"); ax.set_xlabel("K candidates"); ax.set_ylabel("NDCG@10")
ax2 = ax.twinx(); ax2.plot(ex2.K, ex2["p50 latency ms"], "s--", color="C1"); ax2.set_ylabel("p50 latency (ms)", color="C1"); plt.title("Retrieval-scoring tradeoff"); plt.show()

**Takeaway.** Latency rises roughly linearly with K (more candidates to score) while NDCG flattens once retrieval recall saturates — the right K is the knee of that curve, not the largest value.

## Exercise 3 — Second candidate source (trending) merged with two-tower retrieval
`ml-latest-small` has no live clock, so "trending" = most-liked items in the most recent `TRENDING_DAYS` of the *training* interactions (no test leakage).

In [ ]:
TRENDING_DAYS = 730
max_ts = train_pos.timestamp.max(); recent = train_pos[train_pos.timestamp >= max_ts - TRENDING_DAYS * 86400]
trend_ids = recent.i.value_counts().head(50).index.values
print(f"trending window: {len(recent)} interactions -> top-{len(trend_ids)} trending items")

def eval_merge(users, K, extra=None):
    nd, cr, tm = [], [], []
    for u in users:
        t = {}; fin, cand, _ = recommend_pipeline(int(u), K=K, timings=t, extra_cands=extra, do_rerank=False)
        nd.append(ndcg_at_k(fin, test_items[u])); cr.append(len(set(cand) & test_items[u]) / len(test_items[u])); tm.append(t["total"])
    return {"NDCG@10": round(np.mean(nd), 4), "test-item recall in candidates": round(np.mean(cr), 4), "p50 ms": round(np.median(tm), 2)}
display(pd.DataFrame({"two-tower only (K=100)": eval_merge(sample_users, 100),
                      "two-tower (K=100) + trending(50)": eval_merge(sample_users, 100, trend_ids),
                      "two-tower only (K=150)": eval_merge(sample_users, 150)}).T)

**Takeaway.** This is a "mixed" hybrid (Task 13, Phase 7): each source covers a different failure mode (personal taste vs. what's hot / what's new). Compare against the **K=150** row for a fair size-matched baseline — if the merged row only matches it, trending adds coverage but not accuracy on this static dataset, which is the expected offline result; its real value (freshness, cold-start) shows up online.

## Exercise 4 — Item cold-start tracing
Pick an existing movie, *pretend it was just added*, and trace which stage fails at each step.

In [ ]:
rs4 = np.random.default_rng(7); x = int(rs4.choice(np.where(pop >= 40)[0]))
print("pretend-new item:", titles[x], "| true train popularity:", pop[x])
unseen_users = np.where(~seen_mask[:, x])[0]
Qu = Qall[unseen_users]

keep = np.array([i for i in range(n_items) if i != x])
inner = faiss.IndexFlatIP(d)                                                                              # keep a reference alive
idmap = faiss.IndexIDMap(inner); idmap.add_with_ids(C[keep], keep.astype("int64"))                        # index built BEFORE the item existed
hits = lambda idx: int((idx.search(Qu, 100)[1] == x).any(axis=1).sum())
before = hits(idmap)

cold = candidate_tower.embed_cold(tf.constant(item_feats[x:x + 1])).numpy().astype("float32")             # content-only embedding
cos_cold_true = float(cold[0] @ C[x] / (np.linalg.norm(cold[0]) * np.linalg.norm(C[x])))
idmap.add_with_ids(cold, np.array([x], dtype="int64"))                                                    # streaming add (no full rebuild)
after = hits(idmap)

_, nn = flat.search(cold, 11); nn = [i for i in nn[0] if i != x][:10]
proxy = S_neumf[:, nn].mean(1)                                                                            # scoring-stage fallback: avg score of its neighbours
corr = float(np.corrcoef(proxy[unseen_users], S_neumf[unseen_users, x])[0, 1])

print(f"users who retrieve it in top-100 | before index update: {before} / {len(unseen_users)} | after add: {after} / {len(unseen_users)}")
print(f"cosine(content-only embedding, fully-trained embedding) = {cos_cold_true:.3f}")
print(f"correlation of neighbour-average proxy score with the true NeuMF score = {corr:.3f}")
display(pd.DataFrame([
    {"stage": "Candidate generation", "problem for a new item": "Not in the ANN index -> can never be retrieved", "what must change": "Compute its candidate-tower embedding from content features (ID vector unknown) and add it to the index (streaming add_with_ids, or wait for the next rebuild)"},
    {"stage": "Scoring", "problem for a new item": "NeuMF has no ID-embedding row for it -> cannot score it", "what must change": "Fallback score (e.g. neighbour-average as above), or content features in the scorer, or grow the embedding table at next retrain"},
    {"stage": "Re-ranking", "problem for a new item": "Nothing breaks, but a low score keeps it off the list", "what must change": "Freshness boost / exploration quota gives it initial exposure to collect interactions"},
    {"stage": "Training/ops", "problem for a new item": "No interactions yet -> ID embedding untrained", "what must change": "After enough impressions, retrain/refresh so the ID embedding carries real signal"}]))

## Exercise 5 — A/B test designs per pipeline stage

| | **Retrieval-model change** (e.g. new two-tower or new ANN index) | **Re-ranking-rule change** (e.g. stronger diversity / tail quota) |
|---|---|---|
| **Hypothesis** | The new retriever surfaces better candidates, so downstream engagement rises | The new rule improves long-term outcomes (retention, discovery, creator health) at acceptable short-term cost |
| **Randomisation unit** | User (consistent assignment); keep scorer & re-ranker identical across arms | User; keep retrieval & scoring identical across arms |
| **Primary metric** | Online engagement per session (CTR / watch-time); offline gate: retrieval **recall@K** and candidate-set coverage | The metric the rule is *for*: 7/28-day retention, catalog coverage / tail-exposure share, new-creator impressions |
| **Guardrails** | p95/p99 latency & ANN timeouts, empty/short-list rate, retrieval<->scoring consistency, share of recommendations the scorer later discards | Short-term CTR / watch-time must not drop beyond a pre-set tolerance; complaint/skip rate; ILD not so high that relevance collapses |
| **Secondary / diagnostic** | Overlap with old candidate set, item coverage, popularity share of candidates, cold-start item exposure | Per-group exposure, repeat-item rate, genre concentration |
| **Duration & power** | Size with `n_per_arm()` above using baseline CTR and MDE; run >= 1-2 full weekly cycles | Longer (retention effects are slow); run >= 2-4 weeks and read retention with holdouts |
| **Ship / kill rule** | Ship if the primary improves with CI excluding 0 and no guardrail breaches | Ship if the target metric improves *and* guardrails stay within tolerance; otherwise tune `lam` / quota and re-test |
| **Pitfall** | Effects are masked or amplified by the scorer; novelty effects fade | Short experiments miss long-term effects; the trade-off is a *product* decision, not purely statistical |

---
# Deliverables Checklist

- [x] **Phase 1:** Three-stage architecture diagram + Tasks 11–14 model mapping
- [x] **Phase 2:** Three-measure comparison table (SVD stand-in **and** two-tower embeddings) + popularity-bias analysis
- [x] **Phase 3:** Trained two-tower model + saved candidate embeddings (`item_embeddings.npy`)
- [x] **Phase 4:** FAISS ANN index + exact-vs-approximate latency/recall tables + 200K-item scale demo + tradeoff statement
- [x] **Phase 5:** Retrieve-then-score vs brute-force full-catalog NeuMF comparison
- [x] **Phase 6:** MMR / freshness / business-rule / fairness re-ranker + ablation table + lambda sweep
- [x] **Phase 7:** Latency budget table + measured end-to-end timing + bottleneck analysis
- [x] **Phase 8:** `get_recommendations()` tested on 3 users + monitoring + A/B design + production readiness summary
- [x] **Exercises:** all 5 attempted
- [x] **Code:** notebook runs top to bottom

*Run **Kernel -> Restart & Run All** once at the end to confirm a clean top-to-bottom run before you submit.*